In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install anndata scvi-tools --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 641.1/641.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 35.4 MB/s eta 0:00:00
  

In [2]:
import os, json
import numpy as np
import anndata as ad

# === inputs ===
HVG_H5AD  = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"
SPLIT_ROOT = "/content/drive/MyDrive/scMILD_residual/splits_donor_holdout"
SEED = 11  # 네가 메인으로 쓰는 시드로 고정

COL_SAMPLE = "sampleID"
COL_DONOR  = "donor_id"
COL_Y      = "CoVID-19 severity"
COL_SRCROW = "_src_row"   # 있으면 같이 저장(무결성/추적용)

OUT_DIR = f"/content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed{SEED}_batchNone"
#os.makedirs(OUT_DIR, exist_ok=True)


In [5]:
import os

SPLIT_DIR = os.path.join(SPLIT_ROOT, f"seed{SEED}")

def read_list_txt(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    out = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            out.append(str(s))
    return out

# donors
train_donors = set(read_list_txt(os.path.join(SPLIT_DIR, "donors_train.txt")))
val_donors   = set(read_list_txt(os.path.join(SPLIT_DIR, "donors_val.txt")))
test_donors  = set(read_list_txt(os.path.join(SPLIT_DIR, "donors_test.txt")))

# samples (optional but useful for cross-checks)
train_samples = set(read_list_txt(os.path.join(SPLIT_DIR, "samples_train.txt")))
val_samples   = set(read_list_txt(os.path.join(SPLIT_DIR, "samples_val.txt")))
test_samples  = set(read_list_txt(os.path.join(SPLIT_DIR, "samples_test.txt")))

print("[OK] donors:", len(train_donors), len(val_donors), len(test_donors))
print("[OK] samples:", len(train_samples), len(val_samples), len(test_samples))

# leakage hard checks
assert len(train_donors & val_donors) == 0, "donor leakage train<->val"
assert len(train_donors & test_donors) == 0, "donor leakage train<->test"
assert len(val_donors & test_donors) == 0, "donor leakage val<->test"

assert len(train_samples & val_samples) == 0, "sample leakage train<->val"
assert len(train_samples & test_samples) == 0, "sample leakage train<->test"
assert len(val_samples & test_samples) == 0, "sample leakage val<->test"


[OK] donors: 127 36 33
[OK] samples: 198 42 42


In [6]:
import numpy as np

obs = adata.obs
donor_all  = obs[COL_DONOR].astype(str).values
sample_all = obs[COL_SAMPLE].astype(str).values

mask_train = np.isin(donor_all,  list(train_donors))
mask_val   = np.isin(donor_all,  list(val_donors))
mask_test  = np.isin(donor_all,  list(test_donors))

# cross-check: train mask에 잡힌 sample들이 train_samples에 포함되는지
bad = set(sample_all[mask_train]) - train_samples
if len(bad) > 0:
    print("[WARN] train donors mask includes samples not listed in samples_train.txt; example:", list(bad)[:10])


In [15]:
adata = ad.read_h5ad(HVG_H5AD, backed="r")
obs = adata.obs

assert COL_DONOR in obs.columns, f"missing {COL_DONOR}"
donor = obs[COL_DONOR].astype(str).values
mask_train = np.isin(donor, list(train_donors))

print("[INFO] total cells:", adata.n_obs)
print("[INFO] train cells:", int(mask_train.sum()))

# train-only: 메모리로 로드 (학습용)
adata_train = adata[mask_train, :].to_memory()

# 안전 체크
for c in [COL_SAMPLE, COL_DONOR, COL_Y]:
    assert c in adata_train.obs.columns, f"missing {c} in train obs"


[INFO] total cells: 914746
[INFO] train cells: 644021


In [29]:
import json, os
cfg = dict(
    seed=0,
    n_latent=32,
    n_layers=2,
    n_hidden=128,
    max_epochs=120,
    accelerator="gpu",
    devices=1,
    early_stopping=True,
    early_stopping_patience=12,
    check_val_every_n_epoch=5,
    train_size=0.90,
    batch_key=None,
)
with open(os.path.join(OUT_DIR, "scvi_config.json"), "w") as f:
    json.dump(cfg, f, indent=2)


In [24]:
os.path.exists(OUT_DIR)

True

In [30]:
import scvi
scvi.settings.seed = 0

scvi.model.SCVI.setup_anndata(adata_train, batch_key=None)

m = scvi.model.SCVI(
    adata_train,
    n_latent=32,          # 필요시 조정
    n_layers=2,
    n_hidden=128,
)

m.train(
    max_epochs=120,
    accelerator="gpu",
    devices=1,
    early_stopping=True,
    early_stopping_patience=12,
    check_val_every_n_epoch=5,
    train_size=0.90,
)

m.save(os.path.join(OUT_DIR, "model"), overwrite=True)
print("[OK] saved model:", os.path.join(OUT_DIR, "model"))


INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training:   0%|          | 0/120 [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=120` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=120` reached.


[OK] saved model: /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/model


In [16]:
import os
import numpy as np

SPLIT_DIR = os.path.join(SPLIT_ROOT, f"seed{SEED}")

def read_list_txt(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

train_donors = set(map(str, read_list_txt(os.path.join(SPLIT_DIR, "donors_train.txt"))))
val_donors   = set(map(str, read_list_txt(os.path.join(SPLIT_DIR, "donors_val.txt"))))
test_donors  = set(map(str, read_list_txt(os.path.join(SPLIT_DIR, "donors_test.txt"))))

train_samples = set(map(str, read_list_txt(os.path.join(SPLIT_DIR, "samples_train.txt"))))
val_samples   = set(map(str, read_list_txt(os.path.join(SPLIT_DIR, "samples_val.txt"))))
test_samples  = set(map(str, read_list_txt(os.path.join(SPLIT_DIR, "samples_test.txt"))))

# leakage hard checks
assert len(train_donors & val_donors) == 0
assert len(train_donors & test_donors) == 0
assert len(val_donors & test_donors) == 0

assert len(train_samples & val_samples) == 0
assert len(train_samples & test_samples) == 0
assert len(val_samples & test_samples) == 0

# build masks (donor 기준)
donor_all  = adata.obs[COL_DONOR].astype(str).values
sample_all = adata.obs[COL_SAMPLE].astype(str).values

mask_train = np.isin(donor_all, list(train_donors))
mask_val   = np.isin(donor_all, list(val_donors))
mask_test  = np.isin(donor_all, list(test_donors))

# cross-check: donor mask로 잡힌 sample들이 sample txt와 일치하는지 검증(강추)
bad_train = set(sample_all[mask_train]) - train_samples
bad_val   = set(sample_all[mask_val])   - val_samples
bad_test  = set(sample_all[mask_test])  - test_samples

if bad_train: print("[WARN] train donor-mask contains samples not in samples_train.txt:", list(bad_train)[:10])
if bad_val:   print("[WARN] val donor-mask contains samples not in samples_val.txt:",   list(bad_val)[:10])
if bad_test:  print("[WARN] test donor-mask contains samples not in samples_test.txt:",  list(bad_test)[:10])


In [19]:
import anndata as ad
import scvi
import numpy as np
import pandas as pd
import os

# === paths ===
HVG_H5AD  = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"
MODEL_DIR = os.path.join(OUT_DIR, "model")   # m.save(...)로 만든 폴더
SPLIT_DIR = os.path.join(SPLIT_ROOT, f"seed{SEED}")

# === columns ===
COL_SAMPLE = "sampleID"
COL_DONOR  = "donor_id"
COL_Y      = "CoVID-19 severity"
COL_SRCROW = "_src_row"   # 있으면 사용, 없으면 자동 무시

# load base adata (backed)
adata = ad.read_h5ad(HVG_H5AD, backed="r")

# load split lists (txt)
def read_list_txt(path):
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

train_donors = set(map(str, read_list_txt(os.path.join(SPLIT_DIR, "donors_train.txt"))))
val_donors   = set(map(str, read_list_txt(os.path.join(SPLIT_DIR, "donors_val.txt"))))
test_donors  = set(map(str, read_list_txt(os.path.join(SPLIT_DIR, "donors_test.txt"))))

donor_all = adata.obs[COL_DONOR].astype(str).values
mask_train = np.isin(donor_all, list(train_donors))
mask_val   = np.isin(donor_all, list(val_donors))
mask_test  = np.isin(donor_all, list(test_donors))

# load model (IMPORTANT: pass adata=None because we'll provide adata in get_latent_representation)
m = scvi.model.SCVI.load(MODEL_DIR, adata=None)
print("[OK] loaded scVI model from:", MODEL_DIR)


INFO     File /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/model/model.pt already        
         downloaded                                                                                                
[OK] loaded scVI model from: /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/model


/usr/local/lib/python3.12/dist-packages/scvi/model/base/_base_model.py:869: UserWarning: Save path contains no saved anndata and no adata was passed. Model will be loaded without anndata.
  ) = _load_saved_files(


In [20]:
def export_latent_for_mask(tag, mask_bool):
    sub = adata[mask_bool, :].to_memory()
    scvi.model.SCVI.setup_anndata(sub, batch_key=None)

    z = m.get_latent_representation(adata=sub).astype(np.float32)
    np.save(os.path.join(OUT_DIR, f"z_{tag}.npy"), z)

    df = sub.obs[[COL_SAMPLE, COL_DONOR, COL_Y]].copy()
    df[COL_SAMPLE] = df[COL_SAMPLE].astype(str)
    df[COL_DONOR]  = df[COL_DONOR].astype(str)
    df[COL_Y]      = df[COL_Y].astype(str)

    if COL_SRCROW in sub.obs.columns:
        df[COL_SRCROW] = sub.obs[COL_SRCROW].astype(np.int64).values

    df = df.rename(columns={COL_SAMPLE:"sampleID", COL_DONOR:"donor_id", COL_Y:"y", COL_SRCROW:"_src_row"})
    df.to_csv(os.path.join(OUT_DIR, f"obs_{tag}.csv"), index=False)

    print(f"[OK] {tag}: z={z.shape}, obs={df.shape}")

export_latent_for_mask("train", mask_train)
export_latent_for_mask("val",   mask_val)
export_latent_for_mask("test",  mask_test)


INFO     Model was loaded without AnnData. Setting up provided AnnData using saved registry.                       
[OK] train: z=(644021, 32), obs=(644021, 4)
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
[OK] val: z=(150216, 32), obs=(150216, 4)
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
[OK] test: z=(120509, 32), obs=(120509, 4)


In [24]:
import os
import pandas as pd

def load_obs(tag):
    df = pd.read_csv(os.path.join(OUT_DIR, f"obs_{tag}.csv"))
    return set(df["donor_id"].astype(str)), set(df["sampleID"].astype(str))

d_tr, s_tr = load_obs("train")
d_va, s_va = load_obs("val")
d_te, s_te = load_obs("test")

print("donor overlap tv:", len(d_tr & d_va), "tt:", len(d_tr & d_te), "vt:", len(d_va & d_te))
print("sample overlap tv:", len(s_tr & s_va), "tt:", len(s_tr & s_te), "vt:", len(s_va & s_te))


donor overlap tv: 0 tt: 0 vt: 0
sample overlap tv: 0 tt: 0 vt: 0


In [25]:
for tag in ["train","val","test"]:
    z = np.load(os.path.join(OUT_DIR, f"z_{tag}.npy"))
    o = pd.read_csv(os.path.join(OUT_DIR, f"obs_{tag}.csv"))
    n = len(o["sampleID"])
    print(tag, "z", z.shape, "obs_n", n, "match", (z.shape[0]==n))


train z (644021, 32) obs_n 644021 match True
val z (150216, 32) obs_n 150216 match True
test z (120509, 32) obs_n 120509 match True


In [26]:
for tag in ["train","val","test"]:
    z = np.load(os.path.join(OUT_DIR, f"z_{tag}.npy"))
    print(tag, "nan", np.isnan(z).any(), "inf", np.isinf(z).any(), "mean", float(z.mean()), "std", float(z.std()))

train nan False inf False mean 0.007353073451668024 std 0.867822527885437
val nan False inf False mean 0.04055105149745941 std 0.8565633296966553
test nan False inf False mean 0.030749093741178513 std 0.8580418229103088


In [29]:
import os
import numpy as np
import pandas as pd

for tag in ["train", "val", "test"]:
    o = pd.read_csv(os.path.join(OUT_DIR, f"obs_{tag}.csv"))
    if "_src_row" in o.columns:
        sr = o["_src_row"].to_numpy()
        print(
            tag,
            "src_row unique", (len(np.unique(sr)) == len(sr)),
            "min", int(sr.min()),
            "max", int(sr.max()),
        )
    else:
        print(tag, "no _src_row column")

train src_row unique True min 0 max 1437017
val src_row unique True min 14767 max 1444465
test src_row unique True min 12835 max 1462698


In [30]:
import pandas as pd
import os

sr_tr = set(pd.read_csv(os.path.join(OUT_DIR, "obs_train.csv"))["_src_row"].astype(int).tolist())
sr_va = set(pd.read_csv(os.path.join(OUT_DIR, "obs_val.csv"))["_src_row"].astype(int).tolist())
sr_te = set(pd.read_csv(os.path.join(OUT_DIR, "obs_test.csv"))["_src_row"].astype(int).tolist())

print("src_row overlap train-val:", len(sr_tr & sr_va))
print("src_row overlap train-test:", len(sr_tr & sr_te))
print("src_row overlap val-test:", len(sr_va & sr_te))


src_row overlap train-val: 0
src_row overlap train-test: 0
src_row overlap val-test: 0


In [31]:
def load_sets(tag):
    df = pd.read_csv(os.path.join(OUT_DIR, f"obs_{tag}.csv"))
    return set(df["donor_id"].astype(str)), set(df["sampleID"].astype(str))

d_tr, s_tr = load_sets("train")
d_va, s_va = load_sets("val")
d_te, s_te = load_sets("test")

print("donor overlap tv/tt/vt:", len(d_tr & d_va), len(d_tr & d_te), len(d_va & d_te))
print("sample overlap tv/tt/vt:", len(s_tr & s_va), len(s_tr & s_te), len(s_va & s_te))


donor overlap tv/tt/vt: 0 0 0
sample overlap tv/tt/vt: 0 0 0


In [32]:
import os, json, hashlib
import numpy as np
import pandas as pd

SEED = 0

def _stable_int_from_str(s: str) -> int:
    # stable across runs/machines (python hash()는 불안정)
    h = hashlib.md5(s.encode("utf-8")).hexdigest()
    return int(h[:8], 16)

def build_bagmap_for_split(out_dir: str, tag: str, cap: int = 4096, seed: int = 0):
    obs_path = os.path.join(out_dir, f"obs_{tag}.csv")
    if not os.path.exists(obs_path):
        raise FileNotFoundError(obs_path)

    df = pd.read_csv(obs_path)

    required = {"sampleID", "donor_id", "y"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{obs_path} missing columns: {missing}")

    n_cells = len(df)
    df["_row"] = np.arange(n_cells, dtype=np.int64)

    bagmap = {}
    rows = []

    # group by sampleID (bag)
    for sid, g in df.groupby("sampleID", sort=False):
        sid_str = str(sid)

        donors = g["donor_id"].astype(str).unique()
        ys     = g["y"].astype(str).unique()
        if len(donors) != 1:
            raise ValueError(f"[{tag}] sampleID={sid_str}: donor_id not single-valued: {donors[:10]}")
        if len(ys) != 1:
            raise ValueError(f"[{tag}] sampleID={sid_str}: y not single-valued: {ys[:10]}")

        donor = donors[0]
        y = ys[0]

        idx_all = g["_row"].to_numpy(dtype=np.int64)
        n_total = int(idx_all.shape[0])

        # deterministic sampling: seed + stable hash(sampleID)
        rng = np.random.default_rng(seed + _stable_int_from_str(sid_str))
        if n_total > cap:
            sel = rng.choice(idx_all, size=cap, replace=False)
            sel.sort()
            idx = sel
        else:
            idx = np.sort(idx_all)
        n_cap = int(idx.shape[0])

        bagmap[sid_str] = {
            "idx": idx.tolist(),     # split-local row indices
            "n_total": n_total,
            "n_cap": n_cap,
            "donor_id": donor,
            "y": y,
        }

        rows.append({
            "split": tag,
            "sampleID": sid_str,
            "donor_id": donor,
            "y": y,
            "n_cells_total": n_total,
            "n_cells_cap": n_cap,
        })

    bagmap_path = os.path.join(out_dir, f"bagmap_{tag}_seed{seed}.json")
    with open(bagmap_path, "w") as f:
        json.dump(bagmap, f)

    sample_df = pd.DataFrame(rows).sort_values(["split", "sampleID"]).reset_index(drop=True)

    print(f"[OK] {tag}: cells={n_cells} bags={len(bagmap)} -> {bagmap_path}")
    return bagmap_path, sample_df

# ---- run for all splits ----
bag_paths = {}
sample_tables = []
for tag in ["train", "val", "test"]:
    p, sdf = build_bagmap_for_split(OUT_DIR, tag, cap=CAP, seed=SEED)
    bag_paths[tag] = p
    sample_tables.append(sdf)

sample_table = pd.concat(sample_tables, axis=0, ignore_index=True)
sample_table_path = os.path.join(OUT_DIR, "sample_table.csv")
sample_table.to_csv(sample_table_path, index=False)
print("[OK] wrote sample_table:", sample_table_path)

# ---- optional sanity: sampleIDs disjoint across splits ----
s_tr = set(sample_table[sample_table["split"]=="train"]["sampleID"])
s_va = set(sample_table[sample_table["split"]=="val"]["sampleID"])
s_te = set(sample_table[sample_table["split"]=="test"]["sampleID"])
print("sample overlap tv/tt/vt:", len(s_tr & s_va), len(s_tr & s_te), len(s_va & s_te))


[OK] train: cells=644021 bags=198 -> /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/bagmap_train_CAP4096_seed0.json
[OK] val: cells=150216 bags=42 -> /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/bagmap_val_CAP4096_seed0.json
[OK] test: cells=120509 bags=42 -> /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/bagmap_test_CAP4096_seed0.json
[OK] wrote sample_table: /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/sample_table.csv
sample overlap tv/tt/vt: 0 0 0


In [3]:
import os, json
import numpy as np
import pandas as pd

paths = {
  "obs_train": os.path.join(OUT_DIR, "obs_train.csv"),
  "obs_val":   os.path.join(OUT_DIR, "obs_val.csv"),
  "obs_test":  os.path.join(OUT_DIR, "obs_test.csv"),
  "z_train":   os.path.join(OUT_DIR, "z_train.npy"),
  "z_val":     os.path.join(OUT_DIR, "z_val.npy"),
  "z_test":    os.path.join(OUT_DIR, "z_test.npy"),
  "bm_train":  os.path.join(OUT_DIR, "bagmap_train_seed0.json"),
  "bm_val":    os.path.join(OUT_DIR, "bagmap_val_seed0.json"),
  "bm_test":   os.path.join(OUT_DIR, "bagmap_test_seed0.json"),
  "sample_table": os.path.join(OUT_DIR, "sample_table.csv"),
}
for k,p in paths.items():
    print(k, os.path.exists(p), p)


obs_train True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/obs_train.csv
obs_val True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/obs_val.csv
obs_test True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/obs_test.csv
z_train True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/z_train.npy
z_val True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/z_val.npy
z_test True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/z_test.npy
bm_train True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/bagmap_train_seed0.json
bm_val True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/bagmap_val_seed0.json
bm_test True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/bagmap_test_seed0.json
sample_table True /content/drive/MyDrive/scMILD_residual/scvi_trainonly/seed11_batchNone/sample_table.c

In [4]:
for tag in ["train","val","test"]:
    z = np.load(os.path.join(OUT_DIR, f"z_{tag}.npy"), mmap_mode="r")
    obs = pd.read_csv(os.path.join(OUT_DIR, f"obs_{tag}.csv"))
    print(tag, "z_rows", z.shape[0], "obs_rows", len(obs), "match", (z.shape[0]==len(obs)))


train z_rows 644021 obs_rows 644021 match True
val z_rows 150216 obs_rows 150216 match True
test z_rows 120509 obs_rows 120509 match True


In [5]:
def check_bagmap(tag, cap=4096):
    obs = pd.read_csv(os.path.join(OUT_DIR, f"obs_{tag}.csv"))
    n = len(obs)
    with open(os.path.join(OUT_DIR, f"bagmap_{tag}_seed0.json"), "r") as f:
        bm = json.load(f)

    # 2-1) bag 수 == sample_table / obs의 sample 수
    n_bags_obs = obs["sampleID"].nunique()
    assert len(bm) == n_bags_obs, (tag, len(bm), n_bags_obs)

    # 2-2) 각 bag 검증
    for sid, rec in bm.items():
        idx = np.asarray(rec["idx"], dtype=np.int64)

        # cap
        if len(idx) > cap:
            raise ValueError(f"{tag}:{sid} cap violated {len(idx)}>{cap}")

        # range
        if idx.size > 0:
            if idx.min() < 0 or idx.max() >= n:
                raise ValueError(f"{tag}:{sid} idx out of range: [{idx.min()},{idx.max()}] vs n={n}")

        # unique + sorted(선택)
        if len(np.unique(idx)) != len(idx):
            raise ValueError(f"{tag}:{sid} duplicate idx")
        if not np.all(idx[:-1] <= idx[1:]):
            raise ValueError(f"{tag}:{sid} idx not sorted")

    print("[OK] bagmap basic integrity:", tag)

for tag in ["train","val","test"]:
    check_bagmap(tag, cap=4096)


[OK] bagmap basic integrity: train
[OK] bagmap basic integrity: val
[OK] bagmap basic integrity: test


In [7]:
def check_bagmap_points_to_correct_sample(tag, cap=4096, n_checks=50):
    obs = pd.read_csv(os.path.join(OUT_DIR, f"obs_{tag}.csv"))
    with open(os.path.join(OUT_DIR, f"bagmap_{tag}_seed0.json"), "r") as f:
        bm = json.load(f)

    sids = list(bm.keys())
    rng = np.random.default_rng(0)
    pick = rng.choice(sids, size=min(n_checks, len(sids)), replace=False)

    for sid in pick:
        idx = np.asarray(bm[sid]["idx"], dtype=np.int64)
        if idx.size == 0:
            continue
        # 해당 idx의 obs.sampleID가 전부 sid인지 확인
        actual = set(obs.iloc[idx]["sampleID"].astype(str))
        if actual != {sid}:
            raise ValueError(f"{tag}:{sid} points to other samples: {list(actual)[:5]}")
    print("[OK] bagmap points-to-sample:", tag)

for tag in ["train","val","test"]:
    check_bagmap_points_to_correct_sample(tag, cap=4096, n_checks=80)


[OK] bagmap points-to-sample: train
[OK] bagmap points-to-sample: val
[OK] bagmap points-to-sample: test


In [8]:
st = pd.read_csv(os.path.join(OUT_DIR, "sample_table.csv"))
for tag in ["train","val","test"]:
    obs = pd.read_csv(os.path.join(OUT_DIR, f"obs_{tag}.csv"))
    with open(os.path.join(OUT_DIR, f"bagmap_{tag}_seed0.json"), "r") as f:
        bm = json.load(f)

    st_sub = st[st["split"]==tag].set_index("sampleID")
    # sampleID set 일치
    assert set(st_sub.index.astype(str)) == set(bm.keys()), f"{tag} sampleID mismatch sample_table vs bagmap"
    # donor/y single-valued in obs
    g = obs.groupby("sampleID")
    assert (g["donor_id"].nunique() == 1).all(), f"{tag} donor_id not single-valued in obs"
    assert (g["y"].nunique() == 1).all(), f"{tag} y not single-valued in obs"

print("[OK] sample_table meta integrity")


[OK] sample_table meta integrity
